# Deep Q-Learning (DQN) — Atari Pong
### Formative 3 Assignment — Training Notebook

**How to use this on Kaggle:**
1. Upload this notebook to Kaggle
2. Enable GPU: Settings → Accelerator → GPU P100
3. Run all cells in order
4. Download the output `dqn_model.zip` from the output panel

---

### What is DQN?
DQN (Deep Q-Network) is a Reinforcement Learning algorithm where:
- The **agent** (our program) watches the game screen
- It tries different actions and receives **rewards** (score +1 for scoring)
- Over thousands of games, it learns which actions lead to more rewards
- The neural network stores this knowledge as **Q-values** (expected future reward for each action)

## Step 1: Install Required Libraries

In [ ]:
# Install everything needed
# This takes about 2 minutes on Kaggle
# IMPORTANT: Install ale-py first to avoid version conflicts
!pip install -q ale-py
!pip install -q stable-baselines3[extra] gymnasium[atari] autorom[accept-rom-license] shimmy

# Accept the Atari ROM license and download ROMs
!AutoROM --accept-license

print('✅ All packages installed!')

## Step 2: Import Libraries

In [ ]:
import os
import ale_py
import gymnasium as gym

# CRITICAL: Register ALE Atari environments with Gymnasium
# Without this line, you get: 'NamespaceNotFound: Namespace ALE not found'
gym.register_envs(ale_py)

from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import VecFrameStack, DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback
import numpy as np
import matplotlib.pyplot as plt

# Verify ALE environments are available
ale_envs = [e for e in gym.envs.registry if 'ALE' in e]
print(f'✅ Libraries imported! Found {len(ale_envs)} ALE environments.')
if len(ale_envs) == 0:
    print('❌ WARNING: No ALE environments found! Restart the kernel and run Step 1 again.')

## Step 3: Configure Your Experiment

**Change these values for each of your 10 experiments!**

| Hyperparameter | What it controls |
|---|---|
| `learning_rate` | How big each learning step is |
| `gamma` | How much the agent values future rewards (0=now only, 1=all future) |
| `batch_size` | How many past experiences to learn from at once |
| `exploration_fraction` | How long to spend in random exploration phase |
| `exploration_final_eps` | Final % of random actions (epsilon at end of training) |

In [ ]:
# ========================================================
# CHANGE THESE VALUES FOR EACH EXPERIMENT (1 through 10)
# ========================================================

EXPERIMENT_NUMBER = 1   # <-- Change this (1-10)

HYPERPARAMS = {
    'learning_rate':         1e-4,   # Try: 1e-3, 5e-4, 1e-4, 5e-5, 1e-5
    'gamma':                 0.99,   # Try: 0.90, 0.95, 0.99, 1.00
    'batch_size':            32,     # Try: 16, 32, 64, 128
    'exploration_fraction':  0.10,   # Try: 0.05, 0.10, 0.20, 0.30
    'exploration_final_eps': 0.05,   # Try: 0.01, 0.05, 0.10
    'buffer_size':           100_000,
    'learning_starts':       10_000,
    'train_freq':            4,
    'target_update_interval': 1000,
}

# Policy: 'CnnPolicy' (looks at pixels) or 'MlpPolicy' (flattened input)
# For Atari visual games, CnnPolicy almost always wins
POLICY = 'CnnPolicy'

# Training duration
# 1_000_000 steps = ~30 min on Kaggle GPU
# Start with 500_000 if you want faster results for testing
TOTAL_TIMESTEPS = 1_000_000

print(f'Experiment #{EXPERIMENT_NUMBER} configured:')
for k, v in HYPERPARAMS.items():
    print(f'  {k}: {v}')

## Step 4: Create the Atari Pong Environment

Why do we wrap the environment?
- The raw game runs at 60fps with full color — too much info for the agent to process
- **AtariWrapper**: converts to grayscale + shrinks image to 84×84 pixels
- **VecFrameStack**: stacks 4 frames together so the agent can "see" movement (like a flip book)

In [ ]:
def make_atari_env():
    def _make():
        env = gym.make('ALE/Pong-v5', render_mode=None)
        env = AtariWrapper(env)  # Grayscale + resize to 84x84
        return env
    
    env = DummyVecEnv([_make])
    env = VecFrameStack(env, n_stack=4)  # Stack 4 frames
    return env

# Test that the environment works
test_env = make_atari_env()
obs = test_env.reset()
print(f'✅ Environment created!')
print(f'   Observation shape: {obs.shape}')  # Should be (1, 84, 84, 4)
print(f'   Action space: {test_env.action_space}')  # How many buttons
test_env.close()

## Step 5: Create and Train the DQN Agent

In [ ]:
# Create environments
env = make_atari_env()
eval_env = make_atari_env()

# Create the DQN agent
model = DQN(
    policy=POLICY,
    env=env,
    learning_rate=HYPERPARAMS['learning_rate'],
    gamma=HYPERPARAMS['gamma'],
    batch_size=HYPERPARAMS['batch_size'],
    exploration_fraction=HYPERPARAMS['exploration_fraction'],
    exploration_final_eps=HYPERPARAMS['exploration_final_eps'],
    buffer_size=HYPERPARAMS['buffer_size'],
    learning_starts=HYPERPARAMS['learning_starts'],
    train_freq=HYPERPARAMS['train_freq'],
    target_update_interval=HYPERPARAMS['target_update_interval'],
    optimize_memory_usage=True,  # Important for Kaggle RAM limits
    verbose=1,
)

print('✅ DQN agent created!')

In [ ]:
# Set up automatic evaluation every 50,000 steps
# This saves the best model found during training
os.makedirs(f'/kaggle/working/models/exp_{EXPERIMENT_NUMBER}', exist_ok=True)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=f'/kaggle/working/models/exp_{EXPERIMENT_NUMBER}/',
    log_path=f'/kaggle/working/logs/exp_{EXPERIMENT_NUMBER}/',
    eval_freq=50_000,
    n_eval_episodes=5,
    deterministic=True,
    render=False,
)

print('🚀 Starting training...')
print(f'   Total steps: {TOTAL_TIMESTEPS:,}')
print('   This will take ~30 minutes on Kaggle GPU...')

# TRAIN!
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=eval_callback,
    progress_bar=True,
)

print('\n✅ Training complete!')

## Step 6: Save the Model

In [ ]:
# Save the final model
model_path = f'/kaggle/working/dqn_model_exp{EXPERIMENT_NUMBER}'
model.save(model_path)
print(f'✅ Model saved to: {model_path}.zip')

# Also save as the main model
model.save('/kaggle/working/dqn_model')
print('✅ Also saved as: /kaggle/working/dqn_model.zip')

env.close()
eval_env.close()

## Step 7: Quick Evaluation — How Well Did the Agent Learn?

In [ ]:
from stable_baselines3.common.evaluation import evaluate_policy

eval_env2 = make_atari_env()

# Run 10 evaluation episodes with greedy policy (no random actions)
mean_reward, std_reward = evaluate_policy(
    model,
    eval_env2,
    n_eval_episodes=10,
    deterministic=True,  # Greedy policy
)

print('\n' + '='*50)
print('  EVALUATION RESULTS (Greedy Policy)')
print('='*50)
print(f'  Mean Reward : {mean_reward:.2f}')
print(f'  Std Reward  : {std_reward:.2f}')
print('='*50)
print('\n  In Pong:')
print('  +21 = Perfect win | 0 = Tie | -21 = Total loss')

if mean_reward > 10:
    print('  🏆 Excellent! Agent is winning consistently.')
elif mean_reward > 0:
    print('  ✅ Good! Agent wins more than it loses.')
elif mean_reward > -10:
    print('  📈 Okay. Agent is competitive but needs more training.')
else:
    print('  📉 Needs more training. Try increasing TOTAL_TIMESTEPS.')

eval_env2.close()

# Save experiment results for your table
print('\n📋 Copy this row into your hyperparameter table:')
print(f"lr={HYPERPARAMS['learning_rate']}, "
      f"gamma={HYPERPARAMS['gamma']}, "
      f"batch={HYPERPARAMS['batch_size']}, "
      f"eps_start=1.0, "
      f"eps_end={HYPERPARAMS['exploration_final_eps']}, "
      f"eps_frac={HYPERPARAMS['exploration_fraction']} "
      f"→ Mean Reward: {mean_reward:.2f}")

## Step 8: Record a Gameplay GIF for Your README

In [ ]:
!pip install -q imageio imageio-ffmpeg

import imageio

def record_gameplay(model_path, output_path, num_steps=500):
    """Record agent gameplay and save as GIF"""
    
    def _make_rgb():
        env = gym.make('ALE/Pong-v5', render_mode='rgb_array')
        env = AtariWrapper(env)
        return env
    
    raw_env = DummyVecEnv([_make_rgb])
    env = VecFrameStack(raw_env, n_stack=4)
    
    model = DQN.load(model_path, env=env)
    frames = []
    obs = env.reset()
    
    for step in range(num_steps):
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, _ = env.step(action)
        
        frame = raw_env.envs[0].render()
        if frame is not None:
            frames.append(frame)
        
        if done[0]:
            obs = env.reset()
    
    env.close()
    imageio.mimsave(output_path, frames, fps=30)
    print(f'✅ Gameplay saved to: {output_path}')

record_gameplay(
    '/kaggle/working/dqn_model',
    '/kaggle/working/gameplay.gif'
)
print('\n📥 Download gameplay.gif from the Kaggle output panel!')

## Hyperparameter Table (fill this in as you run experiments)

| # | Member | lr | gamma | batch | eps_start | eps_end | eps_frac | Mean Reward | Notes |
|---|--------|-----|-------|-------|-----------|---------|----------|-------------|-------|
| 1 | You | 1e-4 | 0.99 | 32 | 1.0 | 0.05 | 0.10 | ? | Baseline |
| 2 | You | 1e-3 | 0.99 | 32 | 1.0 | 0.05 | 0.10 | ? | Higher LR |
| 3 | You | 5e-5 | 0.99 | 32 | 1.0 | 0.05 | 0.10 | ? | Lower LR |
| 4 | You | 1e-4 | 0.95 | 32 | 1.0 | 0.05 | 0.10 | ? | Lower gamma |
| 5 | You | 1e-4 | 0.99 | 64 | 1.0 | 0.05 | 0.10 | ? | Larger batch |
| 6 | You | 1e-4 | 0.99 | 16 | 1.0 | 0.05 | 0.10 | ? | Smaller batch |
| 7 | You | 1e-4 | 0.99 | 32 | 1.0 | 0.01 | 0.10 | ? | Less final exploration |
| 8 | You | 1e-4 | 0.99 | 32 | 1.0 | 0.05 | 0.30 | ? | More exploration time |
| 9 | You | 1e-4 | 0.90 | 32 | 1.0 | 0.05 | 0.10 | ? | Much lower gamma |
| 10 | You | 2e-4 | 0.99 | 64 | 1.0 | 0.02 | 0.15 | ? | Best guess combo |

### How to interpret results:
- **Reward in Pong**: +21 = perfect win, -21 = total loss
- **Good result**: Mean reward > 0 (winning more than losing)